# SAP → TFL Shell Generation Agent | Jupyter Lab / Colab MVP

**Notebook 是工作台，算法位于独立的 `tfl_agent` Python 包。** 按顺序运行 Step 1–5；每一阶段都可以查看结果、证据和错误信息。

- 默认使用 **synthetic fixture**，不调用外部模型、不产生 API 费用。
- 真实运行：修改下一格中的 `MODE` 和输入路径，配置 `OPENAI_API_KEY`，并使用真实的 Template Library。
- 当前仅支持 **Table**；Word 模板必须带有经过验证的 `{{...}}` 占位符。
- 证据能引用原文不代表医学统计解释一定正确；临床研究场景仍须人工审核。

**安装（在项目根目录运行一次）：** `python -m pip install -e ".[notebook,test]"`，然后运行 `jupyter lab`。

## 0. 环境准备 — 本地 Jupyter Lab / Google Colab

**务必先运行下一格，再运行配置与 `from tfl_agent import TFLSession`。**

- **本地 Jupyter Lab**：保持 Notebook 在 `TFL_Agent_Jupyter_MVP/notebooks/`，完整项目就在它上一级。
- **Google Colab**：打开 Notebook 不等于上传 Python 包；云端若只有 `/content/sample_data`，下一格会弹出文件选择器，请上传 **整个 ZIP**（不是仅 `.ipynb`）。
- 第一次运行将自动安装本地代码包，并将工作目录切换到项目根目录。后续 Cell 无需移动文件。


In [ ]:
# 0. 环境准备（这是第一个需要运行的 Code Cell）
# 本地 Jupyter Lab：自动定位项目目录；Colab：缺少项目时上传完整 ZIP。
from pathlib import Path
import os, sys, subprocess, importlib.util, zipfile

def find_project():
    here = Path.cwd().resolve()
    candidates = [here, *here.parents, here / "TFL_Agent_Jupyter_MVP",
                  here.parent / "TFL_Agent_Jupyter_MVP",
                  Path("/content/TFL_Agent_Jupyter_MVP")]
    for folder in candidates:
        if (folder / "pyproject.toml").is_file() and (folder / "src/tfl_agent/__init__.py").is_file():
            return folder
    return None

PROJECT_ROOT = find_project()
if PROJECT_ROOT is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "没找到项目根目录。请解压完整 ZIP，在项目根目录或 notebooks/ 启动 Jupyter Lab。"
        ) from exc

    print("Colab 云端尚无 Python 代码包，请选择完整的 TFL_Agent_Jupyter_MVP ZIP（不是单独的 ipynb）。")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise FileNotFoundError("没有上传 ZIP，请上传完整项目压缩包。")
    archive = Path("/content") / zip_names[0]
    with zipfile.ZipFile(archive) as zf:
        base = Path("/content").resolve()
        for info in zf.infolist():
            target = (base / info.filename).resolve()
            if not target.is_relative_to(base):
                raise ValueError("ZIP 包中包含不安全的文件路径")
        zf.extractall(base)
    PROJECT_ROOT = find_project()
    if PROJECT_ROOT is None:
        raise FileNotFoundError("ZIP 解压后仍找不到项目。请检查压缩包是否包含 pyproject.toml 和 src/tfl_agent/。")

# 保证安装到当前 Notebook Kernel 对应的 Python，而不是另一个环境。
if importlib.util.find_spec("tfl_agent") is None or importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
                           str(PROJECT_ROOT) + "[notebook]"])

# 加入源码路径，允许当前 Kernel 直接导入，不依赖重启 Kernel。
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("src/tfl_agent exists:", (PROJECT_ROOT / "src/tfl_agent/__init__.py").exists())


## 0. Config — 只改这里的参数

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, FileLink
from tfl_agent import TFLSession
from tfl_agent.tools.llm_client import FixtureJSONClient, OpenAIJSONClient

# Supports launching Jupyter from project root OR the notebooks/ directory.
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
assert (PROJECT_ROOT / "pyproject.toml").exists(), "请先在项目根目录运行 jupyter lab"

MODE = "fixture"                     # "fixture" / "api"
MODEL = "gpt-4.1-mini"                # API mode only
SAP_PATH = PROJECT_ROOT / "input" / "synthetic_sap.pdf"
LIBRARY_PATH = PROJECT_ROOT / "library"
SCHEMA_PATH = PROJECT_ROOT / "schemas" / "TFL_Specification.schema.json"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
OUTPUT_DIR = PROJECT_ROOT / "output" / "notebook_demo"
FIXTURE_PATH = PROJECT_ROOT / "examples" / "demo_llm_output.json"

# Content List 可选。指定列表时 TFL_ID/TITLE 设为 None，使用 TFL_INDEX 选择条目。
CONTENT_LIST_PATH = None   # e.g. PROJECT_ROOT / "examples" / "content_list.json"
TFL_ID = "14.2.1"
TFL_TITLE = "Summary of Overall Survival"
TFL_INDEX = 0
ALLOW_REVIEW_DRAFT = False           # 默认禁止未完成统计规格进入 Word 生成

print(f"Project: {PROJECT_ROOT}")
print(f"Mode: {MODE}; SAP: {SAP_PATH.name}; Output: {OUTPUT_DIR}")

## 1. 初始化 Python Agent（不进行抽取）

Notebook 只传参和展示结果。实际处理逻辑来自 `src/tfl_agent/`。如果 `MODE="api"`，请先在本机终端中设置 `OPENAI_API_KEY`，然后启动 Jupyter。不要把 API Key 写进 Notebook。

In [ ]:
if MODE == "fixture":
    llm = FixtureJSONClient(FIXTURE_PATH)
elif MODE == "api":
    llm = OpenAIJSONClient(model=MODEL)
else:
    raise ValueError("MODE must be 'fixture' or 'api'")

session = TFLSession(
    sap_path=SAP_PATH,
    library_path=LIBRARY_PATH,
    schema_path=SCHEMA_PATH,
    prompts_dir=PROMPTS_DIR,
    output_dir=OUTPUT_DIR,
    llm=llm,
    content_list_path=CONTENT_LIST_PATH,
    tfl_id=TFL_ID,
    title=TFL_TITLE,
    tfl_index=TFL_INDEX,
    allow_review=ALLOW_REVIEW_DRAFT,
)
print("TFLSession ready")

## Step 1 — Input Parsing

读取 SAP PDF、解析页码和文本块；有 Content List 则以其为目标清单，没有时按显式 TFL ID + Title 处理。

In [ ]:
parsed = session.step1_parse()
display(pd.DataFrame([parsed]))
print("Selected target:", session.tfl)

In [ ]:
# 查看 SAP 原文块（保留 block_id + 页码，供 Step 2 引用）
display(pd.DataFrame(session.sap["blocks"])[["block_id", "page", "section", "text"]].head(12))

## Step 2 — Specification Building（重点）

**Locate → Extract → Structure**：LLM 提出检索关键词，BM25 召回相关 SAP 文本块，LLM 抽取统计要求，每个字段必须给出可核验原文证据。

如果某项要求没有通过证据核验，系统会标注 issue 并阻止不安全的 Word 生成。

In [ ]:
spec = session.step2_build_specification()
print("TFL:", spec["identity"])
print("Readiness:", spec["readiness"])
print("Schema/evidence check:", session.spec_validation)
display(pd.DataFrame([spec["statistical_requirements"]]).T.rename(columns={0: "extracted value"}))
print("Issues:")
display(pd.DataFrame(spec["issues"]))

In [ ]:
# Step 2 调试：查看定位查询、召回段落和字段来源
debug = session.spec_debug
print("Queries:", debug["queries"])
print("Retrieved SAP blocks:", len(debug["candidates"]))
display(pd.DataFrame(debug["candidates"])[["block_id", "page", "section", "text"]].head(15))
links = spec["provenance"]["field_links"]
display(pd.DataFrame(links)[["json_pointer", "ref_ids", "derivation"]].head(30))

In [ ]:
# 逐字段查看来源证据（Source Block / Page / 原文片段），便于审核 Step 2
source_evidence = {e["ref_id"]: e for e in spec["provenance"]["evidence"]}
evidence_rows = []
for link in spec["provenance"]["field_links"]:
    for ref_id in link["ref_ids"]:
        ev = source_evidence.get(ref_id, {})
        evidence_rows.append({
            "spec_field": link["json_pointer"],
            "source_ref": ref_id,
            "pdf_page": ev.get("page_numbers"),
            "source_section": ev.get("section"),
            "source_text": (ev.get("exact_text") or "")[:240],
        })
display(pd.DataFrame(evidence_rows).head(40))

## Step 3 — Template Retrieval

使用 `Keyword_Index.json` + `Template_Metadata.json` 选择 Table 模板。候选评分都被保存，低分或打平不会强制选择。

In [ ]:
retrieval = session.step3_retrieve_templates(top_k=5)
print("Selected template ID:", retrieval["selected_template_id"])
display(pd.DataFrame(retrieval["candidates"]))

## Step 4 — Shell Generation

首先**预览修改计划**，然后调用 Word 生成模块。当前不允许随意修改没有可靠定位信息的 DOCX。

In [ ]:
plan_preview = session.preview_modification_plan()
print("Safe to generate:", plan_preview.get("safe_to_generate"))
print("Missing placeholders:", plan_preview.get("missing_placeholders", []))
print("Unexpected placeholders:", plan_preview.get("unexpected_placeholders", []))
print("Planned replacements:")
display(pd.DataFrame([
    {"placeholder": k, "new_text": v}
    for k,v in plan_preview.get("replacements", {}).items()
]))

In [ ]:
generation = session.step4_generate_shell()
print(json.dumps(generation, indent=2, ensure_ascii=False))

## Step 5 — Validation & Output

校验 Specification Schema、SAP 证据引用、Word 标题、统计行、治疗组与脚注等内容，并导出 Inventory、Rationale 和 Report。

In [ ]:
report = session.step5_validate_and_export()
print(json.dumps(report, indent=2, ensure_ascii=False))

In [ ]:
print("Output directory:", OUTPUT_DIR)
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        print(" -", f.relative_to(OUTPUT_DIR))
if session.shell_path:
    display(FileLink(session.shell_path, result_html_prefix="Open generated DOCX: "))
else:
    print("No DOCX produced; inspect the readiness and validation reports.")

---
### 换成真实 Study 的方法

1. 将 `MODE` 改为 `"api"`，并从环境变量读取模型 API Key。
2. 设置 `SAP_PATH` 指向真实的文字型 SAP PDF。
3. 将 `LIBRARY_PATH` 改为已审核的模板库路径，`SCHEMA_PATH` 改为你批准的目标 Specification Schema。
4. 可选设置 `CONTENT_LIST_PATH`；如果没有，就提供 `TFL_ID` 与 `TFL_TITLE`。
5. 从 Notebook 顶部重新运行全部单元格，并重点审核 Step 2 的 evidence 与 Step 4 的 modification plan。

**限制：** Synthetic fixture 是预先写好的示例输出，不能用它评估真实临床 SAP 的抽取准确性。真实模板如没有占位符，需要先在独立的模板适配版本中建立并验证 DOCX 定位映射；不要修改原始来源库。